# S6: Combining PEFT Methods (Optional)

**Leaf-JEPA IRP** | Stage 5 — PEFT Adaptation Experiments


**Research Role:** Tests whether different PEFT methods are complementary or redundant.

- **LoRA + VPT-Shallow:** Attention modification + context injection
- **LoRA + BitFit:** Rank decomposition + bias calibration

## Imports & Configuration

In [1]:
import sys
import json
import copy
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast

from stage5_peft_adaptation_experiments.config_stage5 import *
from stage5_peft_adaptation_experiments.peft_utils import (
    set_seed, get_device, load_ijepa_encoder, inject_lora,
    PEFTClassifier, VPTShallowEncoder, count_params, print_param_summary,
    build_dataloaders, load_class_weights, train_one_epoch, evaluate,
    WarmupCosineScheduler, EarlyStopping, save_results,
    load_results, aggregate_seed_results
)

verify_config()
device = get_device()

SET6_DIR = PEFT_DIR / "set6"
SET6_DIR.mkdir(parents=True, exist_ok=True)

Stage 5 Configuration Verification
  ✅  PlantVillage root exists
  ✅  PlantDoc root exists
  ✅  Splits directory exists
  ✅  Normalisation stats exist
  ✅  NORM_MEAN matches JSON
  ✅  NORM_STD matches JSON
  ✅  I-JEPA checkpoint exists
  ❌  Leaf-JEPA checkpoint exists
       → Run Stage 4 first, then update LEAF_JEPA_CHECKPOINT in config_stage5.py
  ✅  Baselines directory exists
  ✅  WANDB_ENTITY set

  9/10 checks passed
  ⚠️  Fix failing checks before running experiments!
Using GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.5 GB


## Combined Model Builders

In [ ]:
def build_lora_vpt_model(checkpoint_path, device, rank=8, num_prompts=50):
    """Build a model with both LoRA and VPT-Shallow applied."""
    # Load and freeze encoder
    encoder = load_ijepa_encoder(
        checkpoint_path, VIT_MODEL_NAME, VIT_EMBED_DIM, device
    )
    
    # 1. Inject LoRA (modifies encoder in-place)
    lora_params = inject_lora(encoder, rank=rank)
    
    # 2. Wrap with VPT-Shallow
    wrapped = VPTShallowEncoder(encoder, num_prompts, VIT_EMBED_DIM)
    
    # Re-enable LoRA params (VPT wrapper re-freezes encoder)
    for name, param in wrapped.encoder.named_parameters():
        if "lora_" in name:
            param.requires_grad = True
    
    model = PEFTClassifier(wrapped, VIT_EMBED_DIM, NUM_CLASSES).to(device)
    return model


def build_lora_bitfit_model(checkpoint_path, device, rank=8):
    """Build a model with both LoRA and BitFit applied."""
    encoder = load_ijepa_encoder(
        checkpoint_path, VIT_MODEL_NAME, VIT_EMBED_DIM, device
    )
    
    # 1. Inject LoRA
    lora_params = inject_lora(encoder, rank=rank)
    
    # 2. Also unfreeze all bias terms
    for name, param in encoder.named_parameters():
        if "bias" in name and not param.requires_grad:
            param.requires_grad = True
    
    model = PEFTClassifier(encoder, VIT_EMBED_DIM, NUM_CLASSES).to(device)
    return model

## Verify Combined Models

In [ ]:
print("Combined Model Verification")
print("=" * 50)

for name, build_fn, kwargs in [
    ("LoRA + VPT-Shallow", build_lora_vpt_model, {"rank": 8, "num_prompts": 50}),
    ("LoRA + BitFit", build_lora_bitfit_model, {"rank": 8}),
]:
    tmp = build_fn(LEAF_JEPA_CHECKPOINT, device, **kwargs)
    print_param_summary(tmp, name)
    del tmp
    torch.cuda.empty_cache()

## Train Combined Models

In [ ]:
combo_results = []

combos = [
    ("LoRA+VPT", build_lora_vpt_model, {"rank": 8, "num_prompts": 50}),
    ("LoRA+BitFit", build_lora_bitfit_model, {"rank": 8}),
]

for combo_name, build_fn, build_kwargs in combos:
    print(f"\n{'='*60}")
    print(f"  COMBINATION: {combo_name}")
    print(f"{'='*60}")
    
    for seed in SUBSET_SEEDS:
        set_seed(seed)
        
        # Build model
        model = build_fn(LEAF_JEPA_CHECKPOINT, device, **build_kwargs)
        params = count_params(model)
        
        # Data
        train_loader, val_loader, test_loader = build_dataloaders(
            PV_ROOT, SPLITS_DIR, NORM_MEAN, NORM_STD,
            fraction=1.0, seed=seed,
            batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        )
        
        # Criterion
        if CLASS_WEIGHTS_PATH and Path(CLASS_WEIGHTS_PATH).exists():
            weights = load_class_weights(CLASS_WEIGHTS_PATH, device)
            criterion = nn.CrossEntropyLoss(weight=weights)
        else:
            criterion = nn.CrossEntropyLoss()
        
        # Optimizer & scheduler
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(
            trainable, lr=BEST_LR["lora"], weight_decay=WEIGHT_DECAY
        )
        
        total_steps = MAX_EPOCHS * len(train_loader)
        scheduler = WarmupCosineScheduler(
            optimizer, int(total_steps * WARMUP_FRACTION), total_steps
        )
        scaler = GradScaler() if USE_AMP else None
        early_stop = EarlyStopping(patience=EARLY_STOP_PATIENCE, mode="max")
        best_state = None
        
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        
        # Train
        for epoch in range(1, MAX_EPOCHS + 1):
            train_m = train_one_epoch(
                model, train_loader, optimizer, criterion, device,
                scaler=scaler, use_amp=USE_AMP,
                gradient_clip=GRADIENT_CLIP, scheduler=scheduler,
            )
            val_m = evaluate(
                model, val_loader, criterion, device, NUM_CLASSES, USE_AMP
            )
            
            if epoch % 10 == 0 or epoch == 1:
                print(f"    Epoch {epoch:3d} | Val F1: {val_m['macro_f1']:.4f}")
            
            if early_stop(val_m["macro_f1"], epoch):
                print(f"    Early stop at epoch {epoch}")
                break
            
            if early_stop.best_epoch == epoch:
                best_state = copy.deepcopy(model.state_dict())
        
        # Evaluate on test set
        if best_state:
            model.load_state_dict(best_state)
        test_m = evaluate(
            model, test_loader, criterion, device, NUM_CLASSES, USE_AMP
        )
        
        peak_vram = (
            torch.cuda.max_memory_allocated() / 1e6
            if device.type == "cuda" else 0
        )
        
        result = {
            "method": combo_name,
            "param_count": params,
            "training_config": {
                "fraction": 1.0,
                "seed": seed,
                "lr": BEST_LR["lora"],
            },
            "results": {
                "test_macro_f1": test_m["macro_f1"],
                "test_accuracy": test_m["accuracy"],
                "best_epoch": early_stop.best_epoch,
            },
            "compute": {"peak_vram_mb": round(peak_vram, 1)},
        }
        combo_results.append(result)
        
        print(f"  ★ {combo_name} seed={seed}: F1 = {test_m['macro_f1']:.4f}")
        
        del model
        torch.cuda.empty_cache()

save_results(combo_results, PEFT_DIR / "set6_combinations.json")
print(f"\n✅ Combination experiments complete: {len(combo_results)} runs")

## Compare Combinations vs Individual Methods

In [ ]:
set1_results = load_results(PEFT_DIR / "set1_method_comparison.json")

# Get individual method scores
individual_scores = {}
for res in set1_results:
    m = res["method"]
    tag = res.get("hp_tag", "")
    key = f"{m}_{tag}"
    if key not in individual_scores:
        individual_scores[key] = []
    individual_scores[key].append(res["results"]["test_macro_f1"])

# Aggregate combo scores
combo_scores = {}
for res in combo_results:
    m = res["method"]
    if m not in combo_scores:
        combo_scores[m] = []
    combo_scores[m].append(res["results"]["test_macro_f1"])

print("\nCombination vs Individual Comparison")
print("=" * 60)

lora_alone = individual_scores.get("lora_r8", [])
vpt_alone = individual_scores.get("vpt_shallow_p50", [])

for combo_name, scores in combo_scores.items():
    combo_mean = np.mean(scores)
    combo_std = np.std(scores)
    
    print(f"\n  {combo_name}:")
    print(f"    Combined:    {combo_mean:.4f} ± {combo_std:.4f}")
    
    if lora_alone:
        lora_mean = np.mean(lora_alone)
        delta = combo_mean - lora_mean
        print(f"    LoRA alone:  {lora_mean:.4f} ± {np.std(lora_alone):.4f}")
        print(f"    Delta:       {delta:+.4f}")
        
        if abs(delta) < 0.005:
            print(f"    → REDUNDANT: combining doesn't meaningfully improve over LoRA alone")
        elif delta > 0.005:
            print(f"    → COMPLEMENTARY: combination outperforms individual methods")
        else:
            print(f"    → INTERFERENCE: combination hurts performance")

print("\n✅ Set 6 complete. All Stage 5 experiments finished.")